# Chunking

In [1]:
import json
import uuid
from pathlib import Path

import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

INPUT_FILE = Path("./extracted_texts.json")
OUTPUT_FILE = Path("./chunks.json")

PARENT_CHUNK_SIZE = 1054
CHILD_CHUNK_SIZE = 256
CHUNK_OVERLAP = 50
ENCODING_NAME = "cl100k_base"

c:\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
parent_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=ENCODING_NAME,
    chunk_size=PARENT_CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

child_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name=ENCODING_NAME,
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP // 2,
    separators=["\n\n", "\n", ". ", " ", ""],
)

enc = tiktoken.get_encoding(ENCODING_NAME)

In [3]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents from {INPUT_FILE}")

Loaded 2 documents from extracted_texts.json


In [4]:
all_chunks = []

for doc in documents:
    doc_id = str(uuid.uuid4())
    filename = doc["filename"]
    full_text = doc["full_text"]

    parent_texts = parent_splitter.split_text(full_text)

    for p_idx, parent_text in enumerate(parent_texts):
        parent_id = str(uuid.uuid4())

        all_chunks.append({
            "chunk_id": parent_id,
            "doc_id": doc_id,
            "filename": filename,
            "text": parent_text,
            "token_count": len(enc.encode(parent_text)),
            "chunk_index": p_idx,
            "parent_id": None,
            "is_parent": True,
        })

        child_texts = child_splitter.split_text(parent_text)
        for c_idx, child_text in enumerate(child_texts):
            all_chunks.append({
                "chunk_id": str(uuid.uuid4()),
                "doc_id": doc_id,
                "filename": filename,
                "text": child_text,
                "token_count": len(enc.encode(child_text)),
                "chunk_index": c_idx,
                "parent_id": parent_id,
                "is_parent": False,
            })

    parent_count = sum(1 for c in all_chunks if c["is_parent"] and c["doc_id"] == doc_id)
    child_count = sum(1 for c in all_chunks if not c["is_parent"] and c["doc_id"] == doc_id)
    print(f"{filename}: {parent_count} parents, {child_count} children")

total_parents = sum(1 for c in all_chunks if c["is_parent"])
total_children = len(all_chunks) - total_parents
print(f"\nTotal: {total_parents} parent chunks, {total_children} child chunks ({len(all_chunks)} total)")

2507.19457v2.pdf: 103 parents, 368 children
2601.11868v1.pdf: 73 parents, 281 children

Total: 176 parent chunks, 649 child chunks (825 total)


In [5]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"Saved {len(all_chunks)} chunks to {OUTPUT_FILE}")

print("\n--- Sample child chunk ---")
sample = next(c for c in all_chunks if not c["is_parent"])
print(f"ID: {sample['chunk_id']}")
print(f"Parent: {sample['parent_id']}")
print(f"Tokens: {sample['token_count']}")
print(f"Text: {sample['text'][:300]}...")

Saved 825 chunks to chunks.json

--- Sample child chunk ---
ID: 0b28f9ec-3d73-4d71-9028-5bdb0899025e
Parent: 0503d079-7223-49eb-a3d7-b0c73a000457
Tokens: 243
Text: Accepted at ICLR 2026 (Oral).
GEPA: REFLECTIVEPROMPTEVOLUTIONCANOUTPER-
FORMREINFORCEMENTLEARNING
Lakshya A Agrawal1, Shangyin Tan1, Dilara Soylu2, Noah Ziems4,
Rishi Khare1,Krista Opsahl-Ong 5,Arnav Singhvi 2,5,Herumb Shandilya 2,
Michael J Ryan2,Meng Jiang 4,Christopher Potts 2,Koushik Sen 1,
Alex...
